# SolverV8 example: multi-pathway 2Q spectrum

Notebook version of `chi7_multiorder_plot_example.py`.

The current configuration uses five interactions (`--+++`), so it computes the chi5-style 2Q example even though the original script name contains chi7.


In [ ]:
from pathlib import Path

import numpy as np
import time
import pickle
import sys
from tqdm.notebook import tqdm
import itertools

from SolverV9 import (
    LiouvilleSpectroscopySolver,
    SpectroscopyPlotter,
    standard_nq_protocol,
)

from IPython.display import display, HTML
display(HTML("<style>div.output_area pre, div.jp-OutputArea-output pre {white-space: pre;}</style>"))

np.set_printoptions(threshold=sys.maxsize, linewidth=np.inf, precision=5, suppress=True)

### Coupled Model

### Solver Parameters

- `T`: System temperature. `T = 0` assumes that the system initially occupies its ground state.
- `Eta`: Homogeneous broadening used in the resolvents, in eV. It primarily controls the spectral linewidth.
- `backend`: Numerical implementation used by the solver. `"dense"` uses dense matrices and NumPy linear-algebra routines.

In [ ]:
solver_params = {
    "T": 0.0,
    "Eta": 0.0001,
    "backend": "dense",
    "parallel_backend": "threading",
    "n_jobs": -1,
}

In [ ]:
hbar = 0.658211951  # in eV fs
kB = 8.617333262 * 1e-5  # Boltzmann constant eV/K
T = 300  # temperature in K
kT = T * kB
beta = 1 / kT

# parameters
scale = 1
scale_en = 1

en_b=1.4907 # bright state excitonic resonant energy in eV
en_d=1.4907 # dark state excitonic resonant energy in eV
detuning = -0.002 * scale_en # detuning between exciton and cavity in eV
en_cav=en_b + detuning # cavity resonant energy in eV

order = 3

rabi = 0.003 * scale  # vaccum rabi splitting
g = rabi / 2  # coupling strength for light-matter interaction in eV
J = 0.00075 / 2 * scale  # coupling strength between heavy and light excitons (from small band gap energy difference)
n_th = 0. # average number of cavity photons

mu_qm1 = 1#0.3  # dipole strength for C
mu_qm2 = 1#0.29  # dipole strength for CC
mu_d1 = 1#0.04  # dipole strength for D
mu_d2 = 1#0.25  # dipole strength for DD

kappa = 0.0011 * scale  # cavity decay strength
gamma_b_decay = 0.0005 * scale  # heavy exciton decay strength
gamma_d_decay = 0.0001 * scale  # light exciton decay strength
gamma_cav_phase = 0.0006 * scale  # cavity dephasing strength
gamma_exc_phase = 0.0001 * scale  # exciton dephasing strength
gamma_bd = 0.001 # b-d swapping strength
gamma_bc = 0.001 # b-c swapping strength

bb_bind=0.00086 * scale_en # strength of BB biexciton binding
bd_bind=0.000 * scale_en # strength of BD biexciton binding
dd_bind=0.000 * scale_en # strength of DD biexciton binding

grid = 100
t2 = 50
model = "rw"  # "no_rw" for no rotating wave approximation, "rw" for rotating wave approximation
# model="no-rw"

date = time.strftime('%Y-%m-%d')
hour = time.strftime("%Hh%Mm%S")

### 0-quantum + 1-quantum + 2-quantum
loop over i for any parameter

In [ ]:
param = "bc_swap_bd_swap"
rang = [0.0001,0.1,5]
for i in tqdm(list(itertools.product(np.geomspace(rang[0], rang[1], rang[2]), np.geomspace(rang[0], rang[1], rang[2])))):
# for i in [0.1]:
    param_name = gamma_bc = i[0]
    param_name2 = gamma_bd = i[1]
    # solver_params = {
    #     "T": 0.0,
    #     "Eta": i,
    #     "backend": "dense",
    #     "parallel_backend": "threading",
    #     "n_jobs": -1,
    # }

    # setting up dm
    exc_basis = np.zeros((10, 10))
    exc_basis[0, 0] = 1 # ground,B,C,D,BB,BC,BD,CC,CD,DD
    rho = exc_basis @ exc_basis.T  # ground state of Hamiltonian

    wc = en_cav / hbar
    wb = en_b / hbar  # bright exciton resonant frequencies
    wd = en_d / hbar  # dark exciton resonant frequencies

    # each individual lowering operators
    b_low = np.zeros_like(exc_basis)
    b_low[0, 1] = 1

    c_low = np.zeros_like(exc_basis)
    c_low[0, 2] = 1

    d_low = np.zeros_like(exc_basis)
    d_low[0, 3] = 1

    bb_low = np.zeros_like(exc_basis)
    bb_low[1, 4] = 1

    bc_low = np.zeros_like(exc_basis)
    bc_low[1, 5] = 1 / np.sqrt(2)

    cb_low = np.zeros_like(exc_basis)
    cb_low[2, 5] = 1 / np.sqrt(2)

    bd_low = np.zeros_like(exc_basis)
    bd_low[1, 6] = 1 / np.sqrt(2)

    db_low = np.zeros_like(exc_basis)
    db_low[3, 6] = 1 / np.sqrt(2)

    cc_low = np.zeros_like(exc_basis)
    cc_low[2, 7] = 1

    cd_low = np.zeros_like(exc_basis)
    cd_low[2, 8] = 1 / np.sqrt(2)

    dc_low = np.zeros_like(exc_basis)
    dc_low[3, 8] = 1 / np.sqrt(2)

    dd_low = np.zeros_like(exc_basis)
    dd_low[3, 9] = 1

    # global lowering operators
    cav_low = c_low + bc_low + cc_low + dc_low
    exc_b_low = b_low + bb_low + cb_low + db_low
    exc_d_low = d_low + bd_low + cd_low + dd_low

    H0_b = b_low.T @ b_low  # b-pop
    H0_c = c_low.T @ c_low  # c-pop
    H0_d = d_low.T @ d_low  # d-pop
    H0_bb = bb_low.T @ bb_low  # bb-pop
    H0_bc = bc_low.T @ bc_low + cb_low.T @ cb_low  # bc-pop
    H0_bd = bd_low.T @ bd_low + db_low.T @ db_low  # bd-pop
    H0_cc = cc_low.T @ cc_low  # cc-pop
    H0_cd = cd_low.T @ cd_low + dc_low.T @ dc_low  # cd-pop
    H0_dd = dd_low.T @ dd_low  # dd-pop

    H0 = hbar * (wb * H0_b + wc * H0_c + wd * H0_d + (2 * wb - bb_bind) * H0_bb + (wb + wc) * H0_bc + (
                wb + wd - bd_bind) * H0_bd + 2 * wc * H0_cc + (wc + wd) * H0_cd + (2 * wd - dd_bind) * H0_dd)

    H_coupling = b_low.T @ d_low  # 1Q B-D coupling
    H_coupling += 2 * (bb_low.T @ bd_low + cb_low.T @ cd_low + db_low.T @ dd_low)  # 2Q B-D coupling
    H_coupling += H_coupling.T

    H_int = b_low.T @ c_low  # 1Q light-matter coupling
    H_int += 2 * (bb_low.T @ bc_low + cb_low.T @ cc_low + db_low.T @ dc_low) # 2Q bright light-matter coupling
    if model == "no-rw":
        H_int += 2 * cav_low @ exc_b_low  # 2Q non-RWA terms (don't conserve energy)
    elif model != "rw":
        print('Wrong model name')
    H_int += H_int.T
    # print(H_int)

    H = H0 + hbar * g * H_int + J * H_coupling  # total hamiltonian
    # print("H0", H0)
    # print("H_int\n", H_int)
    # print("H_coupling\n", H_coupling)
    # print("H", H)
    # print(np.linalg.eig(H))

    ad = mu_qm2 * cav_low + (mu_qm1 - mu_qm1) * c_low + mu_d2 * exc_d_low + (mu_d1 - mu_d2) * d_low  # lowering operator
    obs = ad + ad.T
    # print("mud", obs)
    # print("ad", ad)

    # collapse operators
    c_cav_rel = np.sqrt(kappa * (n_th + 1)) * cav_low # cavity relaxation
    c_cav_exc = np.sqrt(kappa * n_th) * cav_low.T # cavity excitation
    c_mat_rel = np.sqrt(gamma_b_decay * (n_th + 1)) * exc_b_low + np.sqrt(gamma_d_decay * (n_th + 1)) * exc_d_low # exciton relaxation
    c_mat_exc = np.sqrt(gamma_b_decay * n_th) * exc_b_low.T + np.sqrt(gamma_d_decay * n_th) * exc_d_low.T # exciton excitation

    c_dep = (np.sqrt(gamma_exc_phase) * (wb * H0_b + wd * H0_d) + # B and D pop
            np.sqrt(gamma_cav_phase) * wc * H0_c + # C pop
            np.sqrt(gamma_exc_phase) * (2 * wb - bb_bind) * H0_bb + # BB pop
            (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_exc_phase)) * (wb + wc) / 2 * H0_bc + # BC pop
            np.sqrt(gamma_exc_phase) * (wb + wd) * H0_bd + # BD pop
            np.sqrt(gamma_cav_phase) * 2 * wc * H0_cc + # CC pop
            (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_exc_phase)) * (wc + wd) / 2 * H0_cd + # CD pop
            np.sqrt(gamma_exc_phase) * (2 * wd - dd_bind ) * H0_dd) # DD pop

    c_b_d = np.sqrt(gamma_bd) * exc_d_low.T @ exc_b_low # bright exciton to dark exciton transition
    # c_d_b = np.sqrt(gamma_bd) * exc_b_low.T @ exc_d_low # dark exciton to bright exciton transition

    c_b_cav = np.sqrt(gamma_bc) * cav_low.T @ exc_b_low # bright exciton to cavity photon transition
    # c_cav_b = np.sqrt(gamma_bc) * exc_b_low.T @ cav_low # cavity photon to bright exciton transition

    c_ops = [c_cav_rel, c_cav_exc, c_dep, c_mat_rel, c_mat_exc, c_b_cav, c_b_d]#, c_cav_b, c_d_b]
    # for i in c_ops:
    #     print()
    #     print(i)

    # solver
    solver = LiouvilleSpectroscopySolver(solver_params)
    solver.feed_model(
        H,
        obs,
        c_ops_raw=c_ops,
        initial_density_matrix=rho,
        density_matrix_basis="site",
    )

    # pathways
    pathways = solver.generate_pathways_with_ufss(
        "-++",
        maximum_manifold=2,
        component="chi3_1q",
    )

    protocol = standard_nq_protocol(
        order=1,
        nq_interval=1,
        detection_interval=3,
        n_interactions=3,
        nq_axis="omega_1q",
        detection_axis="omega_emit",
    )

    [(pathway.name, pathway.interactions, pathway.coherence_orders) for pathway in pathways]

    # generate the spectrum
    omega_1q = np.linspace(-1.4925, -1.485, grid)
    omega_emit = np.linspace(1.485, 1.4925, grid)

    delays = {
        "t2": t2
    }

    result = solver.generate_NQ_spectrum(
        1,
        protocol,
        axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
        delays=delays,
        pathways=pathways
        )

    save_pdf = True
    output_directory = Path.cwd() / "Result_Test" / "Multiorder_pathway_plot" / date / f"{hour}_{param}{rang}"
    output_directory.mkdir(parents=True, exist_ok=True)

    plotter = SpectroscopyPlotter(detection_phase=np.pi/2)
    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="real",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        # spectrum_pdf_name=f"chi3_Q1_real_{param}{param_name:.6f}.png",
        spectrum_pdf_name=f"chi3_Q1_real_{param}{param_name:.4f}_{param_name2:.4f}.png",
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="imag",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        # spectrum_pdf_name=f"chi3_Q1_imag_{param}{param_name:.6f}.png",
        spectrum_pdf_name=f"chi3_Q1_imag_{param}{param_name:.4f}_{param_name2:.4f}.png",
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="abs",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        # spectrum_pdf_name=f"chi3_Q1_abs_{param}{param_name:.6f}.png",
        spectrum_pdf_name=f"chi3_Q1_abs_{param}{param_name:.4f}_{param_name2:.4f}.png",
        show=False,
    )

    plotter.plot_spectrum_result_contours(
        result,
        spectra="pathways",
        names=["R1", "R2", "R3"],
        totals="selected",
        normalization="panel",
        color_map="PuOr",
        levels=25,
        show=False,
    )
    print(f"{i} diagram done\n")

In [ ]:
with open(output_directory+"/parameters.txt", "w", encoding="utf-8") as file:
    file.write(hour+"\n")
    file.write(f"Energies  bright={en_b:.5f}, dark={en_d:.5f}, cavity={en_cav:.5f}\n")
    file.write(f"detuning delta={detuning:.4}\n")
    file.write(f"light-matter coupling g={g:.4f}\n")
    file.write(f"bright-dark coupling J={J:.4f}\n")
    file.write(f"dephasing  excitons={gamma_exc_phase:.4f}, cavity={gamma_cav_phase:.4f}\n")
    file.write(f"decay  bright={gamma_b_decay:.4f}, dark={gamma_d_decay:.4f}, cavity={kappa:.4f}\n")
    file.write(f"binding energies  bright-bright={bb_bind:.4f}, bright-dark={bd_bind:.4f}, dark-dark={dd_bind:.4f}\n")
    file.write(f"swapping strength  bright-dark={gamma_bd:.4f}, bright-cavity={gamma_bc:.4f}\n")
    file.write(f"grid size(f1,f3)={grid}\n")
    file.write(f"population time={t2}\n")
    file.write(f"model={model}\n")

In [ ]:
# print("Plotted panels:", plot_result.panel_names)
# print("Matching UFSS diagrams:", tuple(plot_result.diagrams))
# if save_pdf:
#     print("Spectrum PDF:", plot_result.spectrum_pdf)
#     print("Diagram PDFs:", plot_result.diagram_paths)
